# 08 — Resolution Speed (RQ6)

**Research Question:** What makes treatment resolution faster?

Three bottlenecks slow kidney stone treatment: diagnostic admissions, long-stay patients,
and geographic access barriers. We investigate each and identify what drives faster resolution.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, save_metrics, CITY_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

Loaded 206,500 admissions, 510 hospitals


## 1. Diagnostic Admissions

Why are patients admitted for imaging that could be outpatient? Hospital-level variation.

In [2]:
diag = kidney[kidney["proc_category"] == "DIAGNOSTIC"].copy()

# Hospital-level diagnostic rates
hosp_diag = kidney.groupby("CNES").apply(
    lambda g: pd.Series({
        "n_total": len(g),
        "n_diag": (g["proc_category"] == "DIAGNOSTIC").sum(),
        "diag_pct": (g["proc_category"] == "DIAGNOSTIC").mean() * 100,
        "diag_mean_los": g[g["proc_category"] == "DIAGNOSTIC"]["DIAS_PERM"].mean()
            if (g["proc_category"] == "DIAGNOSTIC").any() else np.nan,
    })
).reset_index()
hosp_diag = hosp_diag[hosp_diag["n_total"] >= 20]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(hosp_diag["diag_pct"], bins=30, color="#DC2626", edgecolor="white", alpha=0.8)
axes[0].set_title("Diagnostic Admission Rate by Hospital")
axes[0].set_xlabel("% diagnostic admissions")
axes[0].set_ylabel("Number of hospitals")

# Worst offenders
worst = hosp_diag.nlargest(10, "diag_pct")
axes[1].barh(range(len(worst)), worst["diag_pct"], color="#DC2626")
axes[1].set_yticks(range(len(worst)))
axes[1].set_yticklabels([f"CNES {c}" for c in worst["CNES"]], fontsize=8)
axes[1].set_title("Top 10 Hospitals by Diagnostic Rate")
axes[1].set_xlabel("% diagnostic")
axes[1].invert_yaxis()

fig.suptitle("Diagnostic Admission Problem", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "diagnostic_problem", prefix="08")

print(f"Hospitals with >50% diagnostic admissions: {(hosp_diag['diag_pct'] > 50).sum()}")
print(f"Hospitals with >90% diagnostic admissions: {(hosp_diag['diag_pct'] > 90).sum()}")

/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_22698/288108620.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hosp_diag = kidney.groupby("CNES").apply(


  Saved: 08_diagnostic_problem.png
Hospitals with >50% diagnostic admissions: 143
Hospitals with >90% diagnostic admissions: 76


## 2. Long-Stay Pareto

Who are the patients staying >7 days? What's different about them?

In [3]:
THRESHOLD = 7
kidney["is_long_stay"] = (kidney["DIAS_PERM"] > THRESHOLD).astype(int)

long_vs_short = kidney.groupby("is_long_stay").agg(
    count=pd.NamedAgg(column="DIAS_PERM", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    mean_age=pd.NamedAgg(column="age", aggfunc="mean"),
    pct_male=pd.NamedAgg(column="is_male", aggfunc="mean"),
    pct_emergency=pd.NamedAgg(column="is_emergency", aggfunc="mean"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    total_bed_days=pd.NamedAgg(column="DIAS_PERM", aggfunc="sum"),
).rename(index={0: "Normal (<=7d)", 1: "Long-stay (>7d)"})

print("Long-stay vs Normal comparison:")
print(long_vs_short.to_string())

# Procedure mix for long-stay patients
long_proc = kidney[kidney["is_long_stay"] == 1]["proc_category"].value_counts(normalize=True) * 100
short_proc = kidney[kidney["is_long_stay"] == 0]["proc_category"].value_counts(normalize=True) * 100

proc_compare = pd.DataFrame({"Long-stay": long_proc, "Normal": short_proc}).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

proc_compare.plot.barh(ax=axes[0])
axes[0].set_title("Procedure Mix: Long-Stay vs Normal")
axes[0].set_xlabel("%")
axes[0].invert_yaxis()

# Age distribution comparison
axes[1].hist(kidney[kidney["is_long_stay"] == 0]["age"], bins=30, alpha=0.6,
             label="Normal", color="#2563EB")
axes[1].hist(kidney[kidney["is_long_stay"] == 1]["age"], bins=30, alpha=0.6,
             label="Long-stay", color="#DC2626")
axes[1].set_title("Age Distribution: Long-Stay vs Normal")
axes[1].set_xlabel("Age")
axes[1].legend()

fig.suptitle("Long-Stay Patient Profile", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "long_stay_profile", prefix="08")

Long-stay vs Normal comparison:
                  count   mean_los   mean_age  pct_male  pct_emergency  mortality    mean_cost  total_bed_days
is_long_stay                                                                                                  
Normal (<=7d)    197170   1.968692  46.136542  0.475387       0.554111   0.001811   846.992614          388167
Long-stay (>7d)    9330  12.786495  50.121651  0.399786       0.795070   0.038264  2231.685094          119298


  Saved: 08_long_stay_profile.png


## 3. Geographic Access

Hub cities, patient migration, and cities without surgical capability.

In [4]:
# Treatment hub cities
city_stats = kidney.groupby("city_name").agg(
    treated=pd.NamedAgg(column="city_name", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    pct_migrated=pd.NamedAgg(column="migrated", aggfunc="mean"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
).sort_values("treated", ascending=False)

# Surgical capability
city_surgical = kidney[kidney["proc_category"] == "SURGICAL"].groupby("city_name").size()
city_stats["has_surgery"] = city_stats.index.isin(city_surgical.index)

# Cities receiving migrated patients
migrated_to = kidney[kidney["migrated"]].groupby("city_name").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_hubs = city_stats.head(15)
top_hubs["treated"].plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Top 15 Treatment Hub Cities")
axes[0].set_xlabel("Admissions treated")
axes[0].invert_yaxis()

migrated_to.head(15).plot.barh(ax=axes[1], color="#7C3AED")
axes[1].set_title("Top 15 Cities Receiving Migrated Patients")
axes[1].set_xlabel("Migrated patients received")
axes[1].invert_yaxis()

fig.suptitle("Geographic Access Patterns", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "geographic_access", prefix="08")

print(f"Cities with surgical capability: {city_stats['has_surgery'].sum()}")
print(f"Cities without surgical capability: {(~city_stats['has_surgery']).sum()}")
print(f"Overall migration rate: {kidney['migrated'].mean()*100:.1f}%")

  Saved: 08_geographic_access.png
Cities with surgical capability: 138
Cities without surgical capability: 167
Overall migration rate: 36.5%


## 4. Procedure Resolution Speed

Which procedures achieve the shortest stays with best outcomes?

In [5]:
proc_resolution = kidney.groupby("proc_name").agg(
    count=pd.NamedAgg(column="proc_name", aggfunc="count"),
    median_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="median"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
).sort_values("median_los")
proc_resolution = proc_resolution[proc_resolution["count"] >= 50]

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(proc_resolution["median_los"], proc_resolution["mortality"] * 100,
                     s=proc_resolution["count"] / 20, alpha=0.7, color="#2563EB")
for idx, row in proc_resolution.iterrows():
    ax.annotate(idx, (row["median_los"] + 0.1, row["mortality"] * 100),
                fontsize=7, alpha=0.8)
ax.set_title("Procedure Speed vs Safety")
ax.set_xlabel("Median LOS (days)")
ax.set_ylabel("Mortality Rate (%)")
fig.tight_layout()
save_plot(fig, "procedure_resolution", prefix="08")

  Saved: 08_procedure_resolution.png


## 5. Admission Type Effect

In [6]:
adm_type = kidney.groupby("is_emergency").agg(
    count=pd.NamedAgg(column="DIAS_PERM", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    median_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="median"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
).rename(index={0: "Elective", 1: "Emergency"})

print("Admission type comparison:")
print(adm_type.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

adm_type["mean_los"].plot.bar(ax=axes[0], color=["#059669", "#DC2626"])
axes[0].set_title("Mean LOS")
axes[0].set_ylabel("Days")
axes[0].tick_params(axis="x", rotation=0)

(adm_type["mortality"] * 100).plot.bar(ax=axes[1], color=["#059669", "#DC2626"])
axes[1].set_title("Mortality Rate")
axes[1].set_ylabel("%")
axes[1].tick_params(axis="x", rotation=0)

adm_type["mean_cost"].plot.bar(ax=axes[2], color=["#059669", "#DC2626"])
axes[2].set_title("Mean Cost")
axes[2].set_ylabel("BRL")
axes[2].tick_params(axis="x", rotation=0)

fig.suptitle("Elective vs Emergency Outcomes", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "admission_type_effect", prefix="08")

Admission type comparison:
               count  mean_los  median_los  mortality    mean_cost
is_emergency                                                      
Elective       89828  1.775037         1.0   0.001291  1070.580940
Emergency     116672  2.982866         2.0   0.005125   785.578467


  Saved: 08_admission_type_effect.png


## Summary Metrics

In [7]:
metrics = {
    "diagnostic_admissions": int(len(diag)),
    "diagnostic_pct": round(len(diag) / len(kidney) * 100, 1),
    "hospitals_gt50pct_diagnostic": int((hosp_diag["diag_pct"] > 50).sum()),
    "long_stay_pct": round(float(kidney["is_long_stay"].mean() * 100), 1),
    "long_stay_bed_day_pct": round(float(
        kidney[kidney["is_long_stay"] == 1]["DIAS_PERM"].sum() / kidney["DIAS_PERM"].sum() * 100
    ), 1),
    "migration_rate": round(float(kidney["migrated"].mean() * 100), 1),
    "cities_with_surgery": int(city_stats["has_surgery"].sum()),
    "emergency_mean_los": round(float(adm_type.loc["Emergency", "mean_los"]), 2),
    "elective_mean_los": round(float(adm_type.loc["Elective", "mean_los"]), 2),
}
save_metrics(metrics, "08_resolution_speed")

print("\n=== RESOLUTION SPEED ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 08_resolution_speed.json

=== RESOLUTION SPEED ===
  diagnostic_admissions: 41487
  diagnostic_pct: 20.1
  hospitals_gt50pct_diagnostic: 143
  long_stay_pct: 4.5
  long_stay_bed_day_pct: 23.5
  migration_rate: 36.5
  cities_with_surgery: 138
  emergency_mean_los: 2.98
  elective_mean_los: 1.78
